# 05 — Peer Messaging Phase C (Issue 7)

Demonstrates the simultaneous peer-to-peer messaging phase:
1. Load data, create agents, assign exposure, build network
2. Run Phase C: citizens generate messages, then reflect on peer messages
3. Verify simultaneous update (messages generated BEFORE reflections)
4. Analyze peer messaging patterns and reflection content

**Covers:** Issue 7 (Peer Messaging Phase C)  
**Depends on:** Issues 5, 6

In [1]:
import os, sys, random
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

from cag.io.survey import load
from cag.abm.agent import SurveyedCitizen, PoliticalAgent
from cag.abm.environment import SurveyedNation
from cag.abm.attributes.opinion import ClimatePolicyID, SURVEY_QUESTIONS
from cag.io.llm import load_api_key

# Attribute maps and IDs
from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)

print("Imports OK")

Imports OK


## 1. Load Data & Build Environment

In [3]:
random.seed(42)
year = 2026

UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

sn = SurveyedNation(
    year=year, place="UK",
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

data = load("../data/yougov_survey_data/YouGovProcessedData_train.csv")

for i in range(len(data)):
    row = data.iloc[i]
    sc = SurveyedCitizen(
        agent_id=row.get('ID', None), environment=sn,
        year_of_birth=year - int(row.get('age', 0)),
        gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
        region_id=RegionID(int(row.get('tprofile_GOR', 0))),
        education_id=EducationID(int(row.get('profile_education_level', 0))),
        income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
        ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
        family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
        ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
        brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
        politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
        selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
        selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
        openness_id=rescale_1_6(int(row.get('Openness', 0))),
        conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
        sdo_id=rescale_1_7(int(row.get('SDO', 0))),
        edo_id=rescale_1_7(int(row.get('EDO', 0))),
        rwa_id=rescale_1_6(int(row.get('RWA', 0))),
        original_survey_data=data.iloc[i],
    )
    sn.agents_active[sc.id] = sc

sn.political_agent_a = PoliticalAgent("agent_a", "pro_climate")
sn.political_agent_b = PoliticalAgent("agent_b", "anti_climate")
sn.assign_political_exposure()
sn.create_network(seed=42)
sn.assign_network_blocks()

api_key = load_api_key("openai", csv_path="../data/api_key.csv")
target_policy = ClimatePolicyID.CARBON_TAX

n_total = len(sn.agents_active)
n_with_neighbors = sum(1 for c in sn.agents_active.values() if len(c.network_neighbors) > 0)
print(f"Total citizens: {n_total}")
print(f"Citizens with network neighbors: {n_with_neighbors}")

Total citizens: 651
Citizens with network neighbors: 651


## 2. Run Phase C: Peer Messaging

Simultaneous update: ALL citizen messages are generated BEFORE any reflections occur.  
Each citizen exchanges messages with at most `k` random network neighbors (default: 3).

In [4]:
# Limit demo to 10 citizens to avoid hundreds of LLM calls
MAX_DEMO = 10

# Temporarily restrict to a small demo set: only citizens with neighbors
citizens_with_neighbors = [c for c in sn.agents_active.values() if len(c.network_neighbors) > 0]
demo_citizens = citizens_with_neighbors[:MAX_DEMO]
demo_ids = {c.id for c in demo_citizens}

# Save full agents_active and replace with demo subset
full_agents = sn.agents_active.copy()
sn.agents_active = {cid: c for cid, c in full_agents.items() if cid in demo_ids}

# Also restrict network_neighbors to only include demo citizens
saved_neighbors = {}
for c in sn.agents_active.values():
    saved_neighbors[c.id] = c.network_neighbors[:]
    c.network_neighbors = [n for n in c.network_neighbors if n.id in demo_ids]

n_demo = len(sn.agents_active)
n_demo_with_neighbors = sum(1 for c in sn.agents_active.values() if len(c.network_neighbors) > 0)
print(f"Demo subset: {n_demo} citizens ({n_demo_with_neighbors} with neighbors in demo)")

result_c = sn.run_peer_messaging(
    policy_id=target_policy, day=1, k_peers=3,
    api_key=api_key, model="gpt-4o-mini",
)

print(f"\n=== Phase C: Peer Messaging ===")
print(f"Citizens who generated messages: {result_c['messages_generated']}")
print(f"Citizens who reflected: {result_c['reflections_count']}")

Demo subset: 10 citizens (9 with neighbors in demo)

=== Phase C: Peer Messaging ===
Citizens who generated messages: 9
Citizens who reflected: 9


## 3. Sample Peer Messages & Reflections

In [5]:
# Sample generated messages
if result_c["sample_messages"]:
    print("--- Sample Peer Messages ---")
    for i, msg in enumerate(result_c["sample_messages"]):
        print(f"\nMessage {i+1} (first 300 chars):")
        print(msg[:300])

# Sample reflections
if result_c["sample_reflections"]:
    print("\n--- Sample Reflections ---")
    for i, ref in enumerate(result_c["sample_reflections"]):
        print(f"\nReflection {i+1} (first 300 chars):")
        print(ref[:300])

--- Sample Peer Messages ---

Message 1 (first 300 chars):
I generally support the idea of a carbon tax on fossil fuels, especially if the revenues are returned to the public through a dividend. It seems like a fair way to encourage more sustainable energy use while also helping to offset costs for families, which is important for maintaining equity. It ali

Message 2 (first 300 chars):
I generally support the idea of a carbon tax on fossil fuels, especially if the revenues are distributed back to the public. It seems like a fair way to hold polluters accountable while also encouraging a shift towards cleaner energy. Plus, if the tax revenues can help offset costs for individuals a

--- Sample Reflections ---

Reflection 1 (first 300 chars):
Hearing my peers' strong support for a carbon tax, especially with the idea of redistributing revenues to the public, resonates deeply with my own values. I appreciate the emphasis on fairness and equity, as I believe that transitioning to a greene

## 4. Simultaneous Update Verification

Verify that no citizen's reflection influenced another citizen's message within the same phase.

In [6]:
# Simultaneous update is guaranteed by the implementation:
# run_peer_messaging() generates ALL messages in Step 2 before any reflections in Step 3.
# This is verified in test_peer_messaging.py::TestRunPeerMessaging::test_simultaneous_update.

# Here we verify the observable consequence: each citizen's reflection references
# messages that were generated by OTHER citizens (not influenced by reflections).
for c in sn.agents_active.values():
    for ref in c.reflections:
        if ref["phase"] == "C":
            for msg in ref["messages_received"]:
                assert isinstance(msg, str) and len(msg) > 0, \
                    f"Agent {c.id} got an empty message"

print("[PASS] All Phase C reflections reference valid peer messages")

[PASS] All Phase C reflections reference valid peer messages


## 5. Messaging Pattern Analysis

Distribution of messages per citizen, neighbor overlap, and message word counts.

In [7]:
# Messaging pattern analysis
msgs_received = {}
for c in sn.agents_active.values():
    n_msgs = sum(len(r["messages_received"]) for r in c.reflections if r["phase"] == "C")
    msgs_received[c.id] = n_msgs

has_reflections = {cid: n for cid, n in msgs_received.items() if n > 0}
no_reflections = {cid: n for cid, n in msgs_received.items() if n == 0}

print(f"Citizens who received messages: {len(has_reflections)}")
print(f"Citizens who received 0 messages: {len(no_reflections)}")
if has_reflections:
    counts = list(has_reflections.values())
    print(f"Messages received: min={min(counts)}, max={max(counts)}, "
          f"mean={sum(counts)/len(counts):.1f}")

# Word count analysis
all_refs = [r for c in sn.agents_active.values() for r in c.reflections if r["phase"] == "C"]
if all_refs:
    word_counts = [len(r["text"].split()) for r in all_refs]
    print(f"\nReflection word counts: min={min(word_counts)}, max={max(word_counts)}, "
          f"mean={sum(word_counts)/len(word_counts):.1f}")

Citizens who received messages: 9
Citizens who received 0 messages: 1
Messages received: min=1, max=2, mean=1.3

Reflection word counts: min=129, max=168, mean=147.9


## 6. Sanity Checks

In [8]:
checks_passed = 0
checks_total = 0

# Check 1: Citizens with no neighbors (in demo) produced no reflection
checks_total += 1
no_neighbor_citizens = [c for c in sn.agents_active.values() if len(c.network_neighbors) == 0]
no_neighbor_ok = all(len(c.reflections) == 0 for c in no_neighbor_citizens)
if no_neighbor_ok:
    print(f"[PASS] {len(no_neighbor_citizens)} citizens with no neighbors have 0 reflections")
    checks_passed += 1
else:
    print(f"[FAIL] Some no-neighbor citizens have reflections")

# Check 2: All reflections have phase="C"
checks_total += 1
all_refs = [r for c in sn.agents_active.values() for r in c.reflections]
phase_ok = all(r["phase"] == "C" for r in all_refs)
if phase_ok:
    print(f"[PASS] All {len(all_refs)} reflections have phase='C'")
    checks_passed += 1
else:
    print(f"[FAIL] Some reflections have wrong phase")

# Check 3: All reflections have correct dict keys
checks_total += 1
expected_keys = {"day", "phase", "text", "messages_received"}
keys_ok = all(set(r.keys()) == expected_keys for r in all_refs)
if keys_ok:
    print(f"[PASS] All reflections have correct dict keys: {expected_keys}")
    checks_passed += 1
else:
    print(f"[FAIL] Some reflections have incorrect keys")

# Check 4: messages_received contains non-empty strings
checks_total += 1
msgs_ok = all(
    isinstance(m, str) and len(m) > 0
    for r in all_refs
    for m in r["messages_received"]
)
if msgs_ok:
    print(f"[PASS] All messages_received contain non-empty strings")
    checks_passed += 1
else:
    print(f"[FAIL] Some messages_received are empty or not strings")

# Check 5: Messages generated count matches citizens with neighbors
checks_total += 1
gen_ok = result_c["messages_generated"] == n_demo_with_neighbors
if gen_ok:
    print(f"[PASS] messages_generated ({result_c['messages_generated']}) == citizens with neighbors ({n_demo_with_neighbors})")
    checks_passed += 1
else:
    print(f"[FAIL] messages_generated ({result_c['messages_generated']}) != citizens with neighbors ({n_demo_with_neighbors})")

print(f"\n{checks_passed}/{checks_total} checks passed")

[PASS] 1 citizens with no neighbors have 0 reflections
[PASS] All 9 reflections have phase='C'
[PASS] All reflections have correct dict keys: {'day', 'messages_received', 'phase', 'text'}
[PASS] All messages_received contain non-empty strings
[PASS] messages_generated (9) == citizens with neighbors (9)

5/5 checks passed


## 7. LLM Call Summary

In [9]:
# Phase C LLM calls:
# - 1 generate_peer_message per citizen with neighbors
# - 1 receive_peer_messages per citizen who received messages
gen_calls = result_c["messages_generated"]
ref_calls = result_c["reflections_count"]
total_calls = gen_calls + ref_calls

print(f"Phase C: {gen_calls} generate + {ref_calls} reflect = {total_calls} LLM calls (demo)")

# Estimate for full population
full_with_neighbors = sum(1 for c in full_agents.values() if len(c.network_neighbors) > 0)
# Rough estimate: most citizens will both generate and receive
est_total = full_with_neighbors * 2
print(f"\nFor full {len(full_agents)}-citizen population ({full_with_neighbors} with neighbors):")
print(f"  Estimated Phase C LLM calls per day: ~{est_total:,}")
print(f"  Over 30 days: ~{est_total * 30:,}")

# Restore full agents and neighbors
for c in full_agents.values():
    if c.id in saved_neighbors:
        c.network_neighbors = saved_neighbors[c.id]
sn.agents_active = full_agents
print(f"\nRestored full population: {len(sn.agents_active)} citizens")

Phase C: 9 generate + 9 reflect = 18 LLM calls (demo)

For full 651-citizen population (650 with neighbors):
  Estimated Phase C LLM calls per day: ~1,300
  Over 30 days: ~39,000

Restored full population: 651 citizens
